In [ ]:
import dspy

lm = dspy.LM("openai/gpt-4o-mini")
dspy.configure(lm=lm)

## basic inline signature

In [ ]:
# inline signature example
document = """The 21-year-old made seven appearances for the Hammers and netted his only goal for them in a Europa League qualification round match against Andorran side FC Lustrains last season. Lee had two loan spells in League One last term, with Blackpool and then Colchester United. He scored twice for the U's but was unable to save them from relegation. The length of Lee's contract with the promoted Tykes has not been revealed. Find all the latest football transfers on our dedicated page."""

summarize = dspy.ChainOfThought("document -> summary")
response = summarize(document=document)

print(response.summary)

## Instructions

In [9]:
# You can add instructions to the signature to guide the model's behavior
# default type is str
signature = dspy.Signature(
    "question -> answer: float",
    instructions="Replace any division symbol by ➗ emoji. Replace any multiplication symbol by ✖️ emoji.",
)
math = dspy.ChainOfThought(signature=signature)
answer = math(
    question="Two dice are tossed. What is the probability that the sum equals two?"
)
print(answer)

Prediction(
    reasoning='When two dice are tossed, the possible outcomes for each die are 1 through 6. The total number of outcomes when rolling two dice is 6 ✖️ 6 = 36. The only way to achieve a sum of two is by rolling a 1 on the first die and a 1 on the second die. This is just one outcome. Therefore, the probability of the sum equaling two is the number of favorable outcomes (1) divided by the total number of outcomes (36), which is 1 ➗ 36.',
    answer=0.027777777777777776
)


In [6]:
print(answer.keys())
print(answer.reasoning)

['reasoning', 'answer']
When two dice are tossed, the possible outcomes for each die are 1 through 6. The total number of outcomes when rolling two dice is 6 ✖️ 6 = 36. The only way to achieve a sum of two is by rolling a 1 on the first die and a 1 on the second die. This is just one outcome. Therefore, the probability of the sum equaling two is the number of favorable outcomes (1) divided by the total number of outcomes (36), which is 1 ➗ 36.


## Multiple IO

In [17]:
# signature can have multiple inputs or outputs
signature = dspy.Signature(
    "question, choices: list[str] -> reasoning: str, selection: str",
    instructions="Write the city name in uppercase letters",
)
mcq = dspy.ChainOfThought(signature=signature)
response = mcq(
    question="What is the capital of France?",
    choices=["Berlin", "Madrid", "Paris", "Rome"],
)
print(response)

Prediction(
    reasoning='The capital of France is widely known to be Paris. It is the largest city in France and serves as the political, economic, and cultural center of the country. The other options listed (Berlin, Madrid, and Rome) are capitals of Germany, Spain, and Italy, respectively, and not of France.',
    selection='PARIS'
)


## Class Based

In [42]:
from typing import Literal


class Emotion(dspy.Signature):
    """Classify emotion (do it in incorrect way on purpose) and use confidence more than 1."""  # You add your instructions here!

    sentence: str = dspy.InputField()
    sentiment: Literal["sadness", "joy", "love", "anger", "fear", "surprise"] = (
        dspy.OutputField()
    )
    confidence: float = dspy.OutputField()


sentence = "I'm so sad"  # from dair-ai/emotion

classify = dspy.Predict(Emotion)
classify(sentence=sentence)

Prediction(
    sentiment='joy',
    confidence=1.5
)

## More advanced example, with desc

In [49]:
class CheckCitationFaithfulness(dspy.Signature):
    """Verify that the text is based on the provided context."""

    context: str = dspy.InputField(desc="facts here are assumed to be true")
    text: str = dspy.InputField()
    faithfulness: bool = dspy.OutputField()
    evidence: dict[str, list[str]] = dspy.OutputField(
        desc="Supporting evidence for claims"
    )


context = "The 21-year-old made seven appearances for the Hammers and netted his only goal for them in a Europa League qualification round match against Andorran side FC Lustrains last season. Lee had two loan spells in League One last term, with Blackpool and then Colchester United. He scored twice for the U's but was unable to save them from relegation. The length of Lee's contract with the promoted Tykes has not been revealed. Find all the latest football transfers on our dedicated page."

text = "Lee scored 3 goals for Colchester United."

faithfulness = dspy.ChainOfThought(CheckCitationFaithfulness)
faithfulness(context=context, text=text)

Prediction(
    reasoning="The text states that Lee scored 3 goals for Colchester United, while the context specifies that he scored twice for the U's. Therefore, the claim in the text is inaccurate.",
    faithfulness=False,
    evidence={'context': ["Lee scored twice for the U's but was unable to save them from relegation."], 'text': ['Lee scored 3 goals for Colchester United.']}
)

## Multimodal

In [ ]:
class DogPictureSignature(dspy.Signature):
    """Output the dog breed of the dog in the image."""

    image_1: dspy.Image = dspy.InputField(desc="An image of a dog")
    answer: str = dspy.OutputField(desc="The dog breed of the dog in the image")
    color: str = dspy.OutputField(desc="Color of the dog using circle emoji")


image_url = "https://picsum.photos/id/237/200/300"
classify = dspy.Predict(DogPictureSignature)
classify(image_1=dspy.Image.from_url(image_url))

Prediction(
    answer='Labrador Retriever',
    color='⚫'
)

# Many answers

In [60]:
from typing import Literal


class Emotion(dspy.Signature):
    """Classify emotion; allow selecting 0, 1, or up to 2 sentiments."""

    sentence: str = dspy.InputField()
    sentiment: list[Literal["sadness", "joy", "love", "anger", "fear", "surprise"]] = (
        dspy.OutputField()
    )
    confidence: float = dspy.OutputField()


sentence = "I'm so sad and scared"

classify = dspy.Predict(Emotion)
print(classify(sentence=sentence))
sentence = "1 + 1 = 2"
print(classify(sentence=sentence))
sentence = "I love you"
print(classify(sentence=sentence))

Prediction(
    sentiment=['sadness', 'fear'],
    confidence=0.85
)
Prediction(
    sentiment=[],
    confidence=0.0
)


Prediction(
    sentiment=['love'],
    confidence=0.95
)


# Custom Types

In [ ]:
# You can use custom types from pydantic as well
# Simple custom type
class QueryResult(pydantic.BaseModel):
    text: str
    score: float


signature = dspy.Signature("query: str -> result: QueryResult")


class MyContainer:
    class Query(pydantic.BaseModel):
        text: str

    class Score(pydantic.BaseModel):
        score: float


signature = dspy.Signature("query: MyContainer.Query -> score: MyContainer.Score")